# 5. Deteksi & Inferensi Gambar Uji DOC (*Day-Old Chick*) Sexing
Target Model: **`model_doc_int8.tflite`** | Kategori: **Class 0 (Female)**, **Class 1 (Male)**, dan **Undefined**

Notebook ini melakukan pengujian inferensi lokal menggunakan **TFLite Interpreter INT8** dengan logika keputusan tri-state:
* **Confidence Threshold ($\ge 80\%$)**: Jika tingkat keyakinan model $< 80\%$, status diklasifikasikan sebagai **`undefined` (Ragu-ragu / Ambigu)**.
* **Blank Frame Threshold ($\sigma_{\text{pixel}} \ge 12.0$)**: Jika citra adalah latar belakang polos atau kosong ($\sigma < 12.0$), status diklasifikasikan sebagai **`undefined` (Frame Kosong)**.
* **Simulasi Aktuator Servo**:
  - `Female (0)` $\rightarrow$ Sudut **`0°`** (Sortir Kiri)
  - `Male (1)` $\rightarrow$ Sudut **`180°`** (Sortir Kanan)
  - `Undefined` $\rightarrow$ Sudut **`90°`** (Posisi Netral / Reject ke penampung evaluasi manual)

## Langkah 1: Import Library, Konfigurasi Threshold, & Load Model `model_doc_int8.tflite`

In [ ]:
import os
import sys
import time
import pathlib
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

# Menentukan root direktori proyek PP2 secara otomatis dan portabel
cwd = pathlib.Path.cwd().resolve()
if (cwd / "PP2" / "data").exists():
    PP2_DIR = cwd / "PP2"
elif (cwd / "data").exists():
    PP2_DIR = cwd
elif (cwd.parent / "data").exists():
    PP2_DIR = cwd.parent
else:
    PP2_DIR = cwd
    for p in [cwd] + list(cwd.parents):
        if (p / "PP2" / "data").exists():
            PP2_DIR = p / "PP2"
            break
        elif (p / "data").exists():
            PP2_DIR = p
            break

DATA_RAW_DIR = PP2_DIR / "data" / "doc"
DATA_AUG_DIR = PP2_DIR / "data" / "doc_augmented"
MODEL_DIR = PP2_DIR / "model"

TFLITE_MODEL_PATH = MODEL_DIR / "model_doc_int8.tflite"
LABELS_PATH = MODEL_DIR / "labels_doc.txt"

# Ambang Batas Logika Keputusan Tri-State
CONFIDENCE_THRESHOLD = 0.80   # Minimal keyakinan 80% untuk keputusan definitif
BLANK_STD_THRESHOLD = 12.0    # Standar deviasi piksel minimal agar bukan frame kosong

CLASS_NAMES = ["Female", "Male"]
if LABELS_PATH.exists():
    with open(LABELS_PATH, "r") as f:
        CLASS_NAMES = [line.strip().split()[-1] for line in f if line.strip()]

print(f"Direktori Proyek PP2   : {PP2_DIR.resolve()}")
print(f"Path Model INT8        : {TFLITE_MODEL_PATH.resolve()}")
print(f"Kelas Model            : {CLASS_NAMES}")
print(f"Confidence Threshold   : {CONFIDENCE_THRESHOLD*100:.0f}%")
print(f"Blank Std Threshold    : {BLANK_STD_THRESHOLD}")

# Inisialisasi TFLite Interpreter
import tensorflow as tf

if not TFLITE_MODEL_PATH.exists():
    print(f"[PERINGATAN] File model {TFLITE_MODEL_PATH.name} belum ditemukan!")
    print("Silakan jalankan Notebook 4 terlebih dahulu untuk melatih dan mengekspor model INT8.")
    interpreter = None
else:
    interpreter = tf.lite.Interpreter(model_path=str(TFLITE_MODEL_PATH))
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]
    in_scale, in_zero_point = input_details["quantization"]
    out_scale, out_zero_point = output_details["quantization"]
    print(f"[OK] Model TFLite INT8 berhasil dimuat!")
    print(f"     Input Shape : {input_details['shape']} | Scale={in_scale:.6f}, ZeroPoint={in_zero_point}")
    print(f"     Output Shape: {output_details['shape']} | Scale={out_scale:.6f}, ZeroPoint={out_zero_point}")

## Langkah 2: Fungsi Deteksi Gambar & Logika Keputusan 3-State (`detect_doc_sex`)
Fungsi ini melakukan center crop 1:1, resize ke $160 	imes 160$, normalisasi, kuantisasi INT8, evaluasi standar deviasi frame kosong, dan penentuan sudut aktuator servo.

In [ ]:
def detect_doc_sex(
    image_input,
    threshold=CONFIDENCE_THRESHOLD,
    blank_std_threshold=BLANK_STD_THRESHOLD,
    show_plot=True,
):
    """
    Mendeteksi jenis kelamin DOC dari citra tunggal dengan logika 3-State:
    - Female  -> Sudut Servo 0°
    - Male    -> Sudut Servo 180°
    - Undefined (Blank / Ambigu) -> Sudut Servo 90°
    """
    if interpreter is None:
        raise RuntimeError("TFLite Interpreter belum diinisialisasi atau model tidak ditemukan.")

    # 1. Pemuatan citra (Path / PIL.Image / np.ndarray)
    if isinstance(image_input, (str, pathlib.Path)):
        image_path = pathlib.Path(image_input)
        if not image_path.exists():
            raise FileNotFoundError(f"File tidak ditemukan: {image_path}")
        img_pil = Image.open(image_path).convert("RGB")
        img_name = image_path.name
    elif isinstance(image_input, Image.Image):
        img_pil = image_input.convert("RGB")
        img_name = "PIL.Image"
    elif isinstance(image_input, np.ndarray):
        img_pil = Image.fromarray(np.uint8(image_input)).convert("RGB")
        img_name = "NumPy.ndarray"
    else:
        raise ValueError("Format input tidak didukung.")

    # 2. Preprocessing Cerdas: Aspect-Preserving Center Crop 1:1 & Resize ke 160x160
    img_pil = ImageOps.exif_transpose(img_pil)
    w, h = img_pil.size
    min_dim = min(w, h)
    left = (w - min_dim) // 2
    top = (h - min_dim) // 2
    img_cropped = img_pil.crop((left, top, left + min_dim, top + min_dim))
    img_160 = img_cropped.resize((160, 160), resample=Image.LANCZOS)
    img_arr = np.array(img_160, dtype=np.float32)

    # 3. Deteksi Blank Frame berbasis Standar Deviasi Piksel
    std_dev = float(np.std(img_arr))
    is_blank = std_dev < blank_std_threshold

    # 4. Kuantisasi INT8 & Inferensi
    t0 = time.time()
    img_norm = (img_arr / 127.5) - 1.0
    img_int8 = np.clip(np.round(img_norm / in_scale + in_zero_point), -128, 127).astype(np.int8)
    img_int8 = np.expand_dims(img_int8, axis=0)

    interpreter.set_tensor(input_details["index"], img_int8)
    interpreter.invoke()
    output_int8 = interpreter.get_tensor(output_details["index"])[0]

    # Dequantize output ke probabilitas float [0, 1]
    prob = (output_int8.astype(np.float32) - out_zero_point) * out_scale
    prob = np.clip(prob, 0.0, 1.0)
    if prob.sum() > 0:
        prob = prob / prob.sum()
    infer_time_ms = (time.time() - t0) * 1000.0

    raw_class_id = int(np.argmax(prob))
    max_confidence = float(prob[raw_class_id])

    # 5. Logika Keputusan Tri-State
    if is_blank:
        status = "Undefined (Frame Kosong)"
        class_id = -1
        class_name = "Undefined"
        servo_angle = 90
        decision_color = "#7f7f7f"
    elif max_confidence < threshold:
        status = f"Undefined (Ragu-ragu / Ambigu < {threshold*100:.0f}%)"
        class_id = -1
        class_name = "Undefined"
        servo_angle = 90
        decision_color = "#e377c2"
    else:
        class_id = raw_class_id
        class_name = CLASS_NAMES[class_id]
        status = f"{class_name} ({max_confidence*100:.1f}%)"
        servo_angle = 0 if class_id == 0 else 180
        decision_color = "#2ca02c" if class_id == 0 else "#1f77b4"

    result = {
        "image_name": img_name,
        "class_id": class_id,
        "class_name": class_name,
        "confidence": max_confidence,
        "probabilities": prob.tolist(),
        "std_dev": std_dev,
        "is_blank": is_blank,
        "status": status,
        "servo_angle": servo_angle,
        "infer_time_ms": infer_time_ms,
    }

    # 6. Visualisasi Hasil
    if show_plot:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5), gridspec_kw={"width_ratios": [1.2, 1]})
        
        # Subplot 1: Citra & Overlay Hasil
        ax1.imshow(img_160)
        ax1.set_title(f"{img_name}\nStd Dev: {std_dev:.1f} | Infer: {infer_time_ms:.1f} ms", fontsize=10)
        ax1.axis("off")
        
        # Subplot 2: Bar Probabilitas & Status Keputusan
        y_pos = np.arange(len(CLASS_NAMES))
        bars = ax2.barh(y_pos, prob, color=["#ff7f0e", "#1f77b4"], alpha=0.85, edgecolor="black")
        ax2.set_yticks(y_pos)
        ax2.set_yticklabels(CLASS_NAMES, fontsize=11, fontweight="bold")
        ax2.set_xlim(0.0, 1.05)
        ax2.axvline(x=threshold, color="red", linestyle="--", label=f"Ambang Batas ({threshold*100:.0f}%)")
        ax2.set_xlabel("Tingkat Keyakinan (Confidence)", fontsize=10)
        ax2.set_title(f"Aksi Servo: {servo_angle}° ({status})", fontsize=11, fontweight="bold", color=decision_color)
        ax2.legend(loc="lower right", fontsize=9)
        
        for bar in bars:
            w_val = bar.get_width()
            ax2.text(w_val + 0.02, bar.get_y() + bar.get_height() / 2, f"{w_val*100:.1f}%", va="center", fontsize=10, fontweight="bold")
            
        plt.tight_layout()
        plt.show()

    return result

## Langkah 3: Uji Inferensi Citra Tunggal DOC
Menguji inferensi satu per satu sampel uji dari folder `data/doc_augmented/test/`.

In [ ]:
test_female_dir = DATA_AUG_DIR / "test" / "Female"
test_male_dir   = DATA_AUG_DIR / "test" / "Male"

# Fallback ke folder mentah jika folder augmented belum dibuat
if not test_female_dir.exists():
    test_female_dir = DATA_RAW_DIR / "Female"
    test_male_dir   = DATA_RAW_DIR / "Male"

sample_female = sorted(list(test_female_dir.glob("*.jpg")))[0] if list(test_female_dir.glob("*.jpg")) else None
sample_male   = sorted(list(test_male_dir.glob("*.jpg")))[0] if list(test_male_dir.glob("*.jpg")) else None

if interpreter is not None:
    if sample_female:
        print("=== PENGUJIAN SAMPEL CITRA BETINA (FEMALE) ===")
        res_f = detect_doc_sex(sample_female, show_plot=True)
        print(f"Hasil: {res_f['status']} | Sudut Servo: {res_f['servo_angle']}°\n")

    if sample_male:
        print("=== PENGUJIAN SAMPEL CITRA JANTAN (MALE) ===")
        res_m = detect_doc_sex(sample_male, show_plot=True)
        print(f"Hasil: {res_m['status']} | Sudut Servo: {res_m['servo_angle']}°\n")
else:
    print("[INFO] Model TFLite belum tersedia di folder model.")

## Langkah 4: Pengujian Batch / Grid (Female, Male, Blank Frame, & Ambigu)
Menguji kehandalan sistem pada 4 kondisi nyata:
1. Citra Betina (Female) $\rightarrow$ Servo $0^\circ$
2. Citra Jantan (Male) $\rightarrow$ Servo $180^\circ$
3. Citra Frame Kosong (Latar datar $\sigma < 12.0$) $\rightarrow$ Servo $90^\circ$ (Reject)
4. Citra Ambigu (Blend 50% Female + 50% Male) $\rightarrow$ Servo $90^\circ$ (Reject)

In [ ]:
if interpreter is not None:
    f_samples = sorted(list(test_female_dir.glob("*.jpg")))[:2]
    m_samples = sorted(list(test_male_dir.glob("*.jpg")))[:2]

    # Citra sintetis Blank Frame
    blank_img = Image.new("RGB", (160, 160), color=(235, 235, 235))

    # Citra sintetis Ambigu (50% Female + 50% Male)
    ambiguous_img = None
    if f_samples and m_samples:
        im1 = Image.open(f_samples[0]).convert("RGB").resize((160, 160))
        im2 = Image.open(m_samples[0]).convert("RGB").resize((160, 160))
        ambiguous_img = Image.blend(im1, im2, alpha=0.50)

    test_cases = []
    if f_samples:
        test_cases.append(("Female #1", f_samples[0]))
    if m_samples:
        test_cases.append(("Male #1", m_samples[0]))
    test_cases.append(("Blank Frame", blank_img))
    if ambiguous_img:
        test_cases.append(("Ambigu (Blend)", ambiguous_img))
    if len(f_samples) > 1:
        test_cases.append(("Female #2", f_samples[1]))
    if len(m_samples) > 1:
        test_cases.append(("Male #2", m_samples[1]))

    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    axes = axes.flatten()

    for idx, (label, item) in enumerate(test_cases):
        res = detect_doc_sex(item, show_plot=False)
        
        if isinstance(item, (str, pathlib.Path)):
            disp = Image.open(item).resize((160, 160))
        else:
            disp = item
            
        ax = axes[idx]
        ax.imshow(disp)
        
        status_txt = f"{label}\n{res['status']}\nServo: {res['servo_angle']}° (Std: {res['std_dev']:.1f})"
        ax.set_title(status_txt, fontsize=10, fontweight="bold")
        ax.axis("off")

    plt.suptitle("Pengujian Grid Komprehensif Logika Keputusan Tri-State DOC Sexing", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("[INFO] Model TFLite belum tersedia untuk pengujian batch.")

## Langkah 5: Evaluasi Batch Akurasi & Throughput pada Seluruh 10% Test Set (200 Citra)
Mengukur latensi rata-rata per citra (ms) dan mengukur tingkat keputusan definitif vs reject/undefined pada 100 citra Female dan 100 citra Male.

In [ ]:
if interpreter is not None and test_female_dir.exists() and test_male_dir.exists():
    test_files = []
    for f in (DATA_AUG_DIR / "test" / "Female").glob("*.jpg"):
        test_files.append((f, 0)) # 0 = Female
    for f in (DATA_AUG_DIR / "test" / "Male").glob("*.jpg"):
        test_files.append((f, 1)) # 1 = Male

    print(f"Menjalankan evaluasi batch pada {len(test_files)} citra test set...")
    
    times = []
    correct_definitive = 0
    total_definitive = 0
    total_undefined = 0
    
    y_true_eval = []
    y_pred_eval = []

    for f_path, true_cls in test_files:
        res = detect_doc_sex(f_path, show_plot=False)
        times.append(res["infer_time_ms"])
        
        if res["class_id"] == -1:
            total_undefined += 1
        else:
            total_definitive += 1
            y_true_eval.append(true_cls)
            y_pred_eval.append(res["class_id"])
            if res["class_id"] == true_cls:
                correct_definitive += 1

    avg_time = np.mean(times)
    fps = 1000.0 / avg_time if avg_time > 0 else 0
    acc_def = (correct_definitive / total_definitive * 100.0) if total_definitive > 0 else 0.0

    print("=" * 65)
    print("  RINGKASAN EVALUASI BATCH TEST SET TFLITE INT8")
    print("=" * 65)
    print(f"  Total Citra Diuji        : {len(test_files)} citra (100 Female + 100 Male)")
    print(f"  Keputusan Definitif      : {total_definitive} citra ({total_definitive/len(test_files)*100:.1f}%)")
    print(f"  Keputusan Undefined/Ragu : {total_undefined} citra ({total_undefined/len(test_files)*100:.1f}%)")
    print(f"  Akurasi Sampel Definitif : {acc_def:.2f}%")
    print(f"  Rata-rata Waktu Inferensi: {avg_time:.2f} ms per citra (~{fps:.1f} FPS)")
    print("=" * 65)
else:
    print("[INFO] Model atau dataset test belum siap untuk evaluasi batch.")